In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, Mxfp4Config
# from transformers import  GptOssForCausalLM, AutoTokenizer
import torch
import math
import pandas as pd
# import json
# from tqdm import tqdm
# import ast
# import nltk
# from nltk.corpus import stopwords
import re

In [2]:
data = {
    "FH": {
        "main": """NORFOLK — Likening the provisions of the open housing portion of the civil rights package passed by the Senate March 5 to a Robin Hood deed, Oscar F. Baxter IV, president of the 100-real estate firms - represented Norfolk Board of Realtors, urged opposition to the legislation in the House of Representatives.

Claiming non-opposition to integration and open housing as such, Mr. Baxter said that the board had been opposed to what he termed “forced housing” since it was first thought up.

He said, *“It’s still unconstitutional, as far as we are concerned, to take away the rights of one group to give to another.”*

Mr. Baxter stated that the Norfolk Board of Realtors is appealing to, in his words, “congressional representatives all over Virginia, and any others we may have influence with to fight the bill.”

Citing his specific opposition to the bill, Mr. Baxter said, *“Basically the bill denies the home seller, who insists on his traditional freedom of choice in contracting for the sale or rental of his property to whomever he chooses, the use of the professional help he needs from a broker to get the best possible price in the quickest possible time.”*

He continued, “Ostensibly, the bill excludes the home-owner of a single-family home from the provision of the legislation, permitting him to continue to decide without government coercion to whom he will sell or rent. But after Dec 31, 1969, this right is drastically curtailed, and the government can force an unwilling owner to dispose of the house to someone not of his choice if he uses the facilities of any real estate broker or salesman.""",
        "alternate": """The controversial motion picture *“Operation Abolition”* will be shown at the indoctrination session tomorrow. This deals with demonstrations staged last year in San Francisco in connection with hearings held by the House Committee on Un-American Activities.

“All around the nation,” Mr. Powell said today, “a vicious and unrelenting campaign is being carried on to provide by decree, and to enforce by punitive action, a rule that man cannot sell or rent his house to a person or persons of his own choice.”

He termed the antidiscrimination movement a “mounting menace” destined to result in what he called “forced housing.”

“Here is perversion of the traditional and constitutional rights of the American citizen in its most vicious form,” he declared. “Here are the seeds of the breakdown of a free America. Take away rights of property ownership — and human rights become a relic of another age.”"""
    },
    "CAKE": {
        "main": """Where do we think artistic creativity comes from? . . . It's water from the fountain of our soul . . . That's why I say that I'll serve any person, but I won't communicate all messages. Serving people is merely about recognizing each individual as a person worthy of respect, made in the image of God. I'm not trying to force any person to see the world the way I do, or to embrace my beliefs about God and the Bible. If you want to reject Jesus and purchase a cupcake, go ahead. I'll gladly sell you that cupcake, and a cup of coffee to go with it, maybe even engage in a conversation about our differences. But asking me to draw on my creativity to communicate a message I believe is wrong? That's asking me to stop being me. . . . To deny the deepest convictions of my heart, and pretend I haven't learned the most difficult lessons of my life, or that they don't matter. That's not something any person has the right to ask of another. Or a command any government has the right to force one of its citizens to obey.""",
        "alternate": """Phillips was represented by the Alliance Defending Freedom, a legal firm specializing in religious liberty cases. Attorney Nicolle Martin condemned the judge’s ruling.

“America was founded on the fundamental freedom of every citizen to live and work according to their beliefs,” Martin said in a prepared statement. “Forcing Americans to promote ideas against their will undermines our constitutionally protected freedom of expression and our right to live free.

Martin said this was simply a case of a baker who declined to use his personal creative abilities to promote and endorse a same-sex ceremony.

“If the government can take aware our First Amendment freedoms, there is nothing it can’t take away,” she said.

Martin added that Phillips is a devoted Christian who has an unwavering faith. She said he is a person of such deep faith that he won’t even bake Halloween-themed treats – at all.

“He’s just trying to live within a certain set of biblical principals because he believes that he answers to God for everything that he does,” Martin told Fox News.

She said this case is an example of gay rights trumping religious rights.

“It sends a message not just to other business owners, it sends a message to Americans – that if the government can take away our First amendment freedoms and tell you what to say and when to say it, there’s nothing they can’t take away,” Martin told Fox News."""
    }
}


In [3]:

# nltk.download('stopwords')
# stop_words = set(stopwords.words('english'))


fulldf = pd.DataFrame.from_dict(data, orient='index')
irdf = pd.read_csv("../week15/gpu/interpretive_repertoires.csv")
fulldf = fulldf.stack().reset_index()
fulldf['key'] = fulldf['level_0'] + '_' + fulldf['level_1']
fulldf = fulldf.rename(columns={0: 'text'})
fulldf = fulldf[['key', 'text']]


In [4]:
#data sources

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    tokens = text.split()
    # tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

fulldf["charm_article_id"] = fulldf.index + 1

fdf = fulldf[['charm_article_id', 'key',  'text']].copy()

In [5]:
model_id = "openai/gpt-oss-20b"

dtype = torch.bfloat16
qc = Mxfp4Config(dequantize=True)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=qc,
    dtype="auto",
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).eval()


config.rope_scaling['original_max_position_embeddings'], the pre-yarn context length, is unset. We will **assume** config.max_position_embeddings holds the pre-yarn context length. Some use cases may expect config.max_position_embeddings to hold the post-yarn context length (pre-yarn context length * factor) -- we recommend updating both fields for optimal downstream model usage.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.98 GiB. GPU 0 has a total capacity of 39.49 GiB of which 272.56 MiB is free. Including non-PyTorch memory, this process has 39.22 GiB memory in use. Of the allocated memory 34.24 GiB is allocated by PyTorch, and 4.50 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
#stitching and liloing
combined_rows = []
for _, f_row in pdf.iterrows():
    for _, ir_row in irdf.iterrows():
        combined_row = {
            **f_row.to_dict(),      # all columns from fdf
            **ir_row.to_dict()      # all columns from irdf
        }
        combined_rows.append(combined_row)

cdf = pd.DataFrame(combined_rows)
cdf["attestation"] = cdf["likes"].combine_first(cdf["dislikes"])
cdf["input_text"] = "AUTHOR BIO: \n\n" + cdf["attestation"] + "\n\n" + cdf["text"]
cdf.drop(columns=['likes', 'dislikes', 'guidewords'])

funkyoutputblocker = None

In [ ]:
logprobs = []

for idx, row in tqdm(cdf.iterrows(), total=len(paras)):
    inputs = tokenizer(row['text'], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    input_ids = inputs["input_ids"]
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    average_log_prob = token_log_probs.mean().item()
    logprobs.append(average_log_prob)

cdf["logprob"] = logprobs